# SIGMAN Colab 一键推理（Drive路径版，无需 .sh）

这个 Notebook 不依赖 `colab_one_click_infer.sh`。
你只要把文件放到 Google Drive 推荐路径后，按顺序运行每个单元格即可。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ===== 你可以只改这里 =====
ASSETS_ZIP = '/content/drive/MyDrive/SIGMAN/assets_bundle.zip'
INPUT_IMAGE = '/content/drive/MyDrive/SIGMAN/input.jpg'
POSE_PATH = ''  # 留空则使用 demo pose
REPO_URL = 'https://github.com/yyvhang/SIGMAN_release.git'
REPO_DIR = '/content/SIGMAN_release'


In [ ]:
import os
required=[ASSETS_ZIP, INPUT_IMAGE]
missing=[p for p in required if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')
print('✅ Files found')


In [ ]:
%%bash
set -e
cd /content
if [ ! -d SIGMAN_release ]; then
  git clone --recursive https://github.com/yyvhang/SIGMAN_release.git
fi


In [ ]:
%%bash
set -e
cd /content/SIGMAN_release
python -m pip install -U pip
pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118
pip install -U xformers --index-url https://download.pytorch.org/whl/cu118
if [ ! -d diff-gaussian-rasterization ]; then
  git clone --recursive https://github.com/ashawkey/diff-gaussian-rasterization
fi
pip install ./diff-gaussian-rasterization
if [ ! -d gaussian-splatting ]; then
  git clone https://github.com/graphdeco-inria/gaussian-splatting.git
fi
pip install git+https://github.com/NVlabs/nvdiffrast
pip install -r requirements.txt


In [ ]:
%%bash
set -e
cd /content/SIGMAN_release
mkdir -p ckpt/autoencoder ckpt/transformer ckpt/sapiens_1b
wget -O ckpt/autoencoder/autoencoder.safetensors https://huggingface.co/Mr-Hang/SIGMAN/resolve/main/autoencoder.safetensors
wget -O ckpt/transformer/transformer.safetensors https://huggingface.co/Mr-Hang/SIGMAN/resolve/main/transformer.safetensors
wget -O ckpt/sapiens_1b/sapiens_1b_epoch_173_torchscript.pt2 https://huggingface.co/facebook/sapiens-pretrain-1b-torchscript/resolve/main/sapiens_1b_epoch_173_torchscript.pt2


In [ ]:
import os, zipfile, shutil
repo='/content/SIGMAN_release'
tmp='/content/_sigman_assets'
if os.path.exists(tmp):
    shutil.rmtree(tmp)
os.makedirs(tmp, exist_ok=True)
with zipfile.ZipFile(ASSETS_ZIP,'r') as zf:
    zf.extractall(tmp)
smplx_dst=os.path.join(repo,'core/modules/deformers/smplx/SMPLX')
template_dst=os.path.join(repo,'core/modules/deformers/template')
os.makedirs(smplx_dst, exist_ok=True)
os.makedirs(template_dst, exist_ok=True)
candidates=[os.path.join(tmp,'smplx','SMPLX'), os.path.join(tmp,'SMPLX')]
smplx_src=next((p for p in candidates if os.path.isdir(p)), None)
if smplx_src is None:
    raise FileNotFoundError('Cannot find smplx/SMPLX or SMPLX in assets zip')
template_src=os.path.join(tmp,'template')
if not os.path.isdir(template_src):
    raise FileNotFoundError('Cannot find template folder in assets zip')
for name in os.listdir(smplx_src):
    s=os.path.join(smplx_src,name); d=os.path.join(smplx_dst,name)
    if os.path.isdir(s):
        shutil.copytree(s,d,dirs_exist_ok=True)
    else:
        shutil.copy2(s,d)
for name in os.listdir(template_src):
    s=os.path.join(template_src,name); d=os.path.join(template_dst,name)
    if os.path.isdir(s):
        shutil.copytree(s,d,dirs_exist_ok=True)
    else:
        shutil.copy2(s,d)
print('✅ assets extracted and copied')


In [ ]:
%%bash
set -e
cd /content/SIGMAN_release/core/modules/deformers
if [ ! -f template/init_uv_smplx_thu.npy ] || [ ! -f template/init_pcd_smplx_thu.npy ] || [ ! -f template/init_rot_smplx_thu.npy ] || [ ! -f template/face_mask_thu.npy ] || [ ! -f template/hands_mask_thu.npy ] || [ ! -f template/outside_mask_thu.npy ]; then
  python preprocess_smplx.py
  python subdivide_smplx.py
  python utils_smplx.py
  python utils_uvpos.py
else
  echo 'template npy exists, skip preprocess'
fi


In [ ]:
import os, subprocess
repo='/content/SIGMAN_release'
pose=POSE_PATH if POSE_PATH else os.path.join(repo,'demo/poses/smplx_demo.npz')
if not os.path.exists(pose):
    raise FileNotFoundError(f'Pose file not found: {pose}')
ckpt_link=os.path.join(repo,'.ckpt')
if not os.path.exists(ckpt_link):
    os.symlink('ckpt', ckpt_link)
cmd=['python','scripts/test_DiT.py','--image_path',INPUT_IMAGE,'--pose_path',pose]
subprocess.run(cmd,cwd=repo,check=True)
print('✅ inference done')


In [ ]:
!ls -lah /content/SIGMAN_release/workspace/outputs
!find /content/SIGMAN_release/workspace/outputs -maxdepth 2 -type f | head -n 30
